 #    Code Python complet pour les méthodes spectrales de Tchebychev 

  Avant de commencer par ecrire les codes( programmes), il me semble judicieux de rappeler que ces codes sont inspires en majorites des livres et articles dans mes References ainsi biensur que des sources en ligne.

 Nous allons donc ici esplorer trois (03) differentes methodes spectrales :
 * Methode de Collocation
 * Methode de Galerkin
 * Methode de Lanczos.
D'abord, nous allons appliquer ces methodes aux EDO Lineaires et dans un second temps, nous allons proposer des applications la "Methode de Collocation" aux EDO non lineaires.
Nous avons pris le soin de ranger toutes ces applications dans des "Exemples" :
* EXEMPLE 1: Equation Lineaire avec solution exactes
* EXEMPLE 2: Equation de Bratu
* EXEMPLE 3: Equation de Diffing(non-lineaire)
* EXEMPLE 4: Probleme avec conditions mixtes(Neumann et Dirichlet)
Enfin de compte un petit programme qui compare les differentes methodes.

In [1]:
### 1. Module de base : fonctions et matrices de Tchebychev
"""
chebyshev_base.py
#Module de base pour les méthodes spectrales de Tchebychev
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve

def chebyshev_points(N, type_points='gauss_lobatto'):
    """
    Génère les points de collocation de Tchebychev
    
    Paramètres:
    -----------
    N : int
        Nombre de points (NBASIS pour les points intérieurs)
    type_points : str
        'gauss_lobatto' : points incluant les bords (xi = cos(pi*j/N))
        'gauss' : points intérieurs uniquement (racines de T_{N+1})
        
    Retourne:
    --------
    x : array
        Points de collocation
    """
    if type_points == 'gauss_lobatto':
        # Points incluant x = ±1
        x = np.cos(np.pi * np.arange(N + 1) / N)
    else:
        # Points intérieurs (type Gauss) - utilisé dans le code Fortran
        x = np.cos(np.pi * np.arange(1, N + 1) / (N + 1))
    return x


def chebyshev_matrix(N, x=None):
    """
    Calcule la matrice de différenciation de Tchebychev
    Algorithme classique de Trefethen
    
    Paramètres:
    -----------
    N : int
        Nombre de points
    x : array, optional
        Points de collocation (calculés automatiquement si None)
        
    Retourne:
    --------
    D : array (N+1, N+1)
        Matrice de différenciation
    x : array
        Points de collocation
    """
    if x is None:
        x = chebyshev_points(N, 'gauss_lobatto')
    
    c = np.ones(N + 1)
    c[0], c[-1] = 2, 2
    c = c * (-1)**np.arange(N + 1)
    
    X = np.tile(x, (N + 1, 1))
    dX = X - X.T
    
    D = np.outer(c, 1/c) / (dX + np.eye(N + 1))
    D = D - np.diag(D.sum(axis=1))
    
    return D, x


def evaluate_chebyshev(x, coeffs):
    """
    Évalue une série de Tchebychev aux points x
    
    Paramètres:
    -----------
    x : array
        Points d'évaluation
    coeffs : array
        Coefficients de la série (a_0, a_1, ..., a_N)
        
    Retourne:
    --------
    y : array
        Valeurs de la série aux points x
    """
    N = len(coeffs) - 1
    y = np.zeros_like(x)
    
    # Utilisation de la récurrence de Clenshaw pour la stabilité
    for xi in x:
        if np.abs(xi) <= 1:
            b_k2 = 0.0
            b_k1 = 0.0
            for k in range(N, 0, -1):
                b_k = 2 * xi * b_k1 - b_k2 + coeffs[k]
                b_k2 = b_k1
                b_k1 = b_k
            # Indice i
            # Récupération de la valeur
            pass
    
    # Version simple (moins stable pour grand N)
    T = np.polynomial.Chebyshev(coeffs)
    return T(x)


def chebyshev_derivative(coeffs):
    """
    Calcule les coefficients de la dérivée d'une série de Tchebychev
    
    Paramètres:
    -----------
    coeffs : array
        Coefficients de la série (c_0, c_1, ..., c_N)
        
    Retourne:
    --------
    coeffs_der : array
        Coefficients de la série dérivée
    """
    N = len(coeffs) - 1
    coeffs_der = np.zeros(N)
    
    if N >= 1:
        coeffs_der[N-1] = 2 * N * coeffs[N]
        for k in range(N-2, -1, -1):
            coeffs_der[k] = coeffs_der[k+2] + 2 * (k+1) * coeffs[k+1]
    
    return coeffs_der


def create_boundary_function(alpha, beta, x):
    """
    Crée une fonction de bord B(x) qui satisfait B(-1)=alpha et B(1)=beta
    
    Paramètres:
    -----------
    alpha, beta : float
        Valeurs aux limites en x = -1 et x = 1
    x : array
        Points d'évaluation
        
    Retourne:
    --------
    B : array
        Valeurs de la fonction de bord
    Bx : array
        Dérivée de la fonction de bord (constante)
    """
    # B(x) = alpha*(1-x)/2 + beta*(1+x)/2
    B = alpha * (1 - x) / 2 + beta * (1 + x) / 2
    Bx = (-alpha + beta) * np.ones_like(x) / 2
    return B, Bx


def create_basis_functions(N, x):
    """
    Crée les fonctions de base φ_j(x) qui s'annulent aux bords
    Basé sur la méthode du code Fortran
    
    Paramètres:
    -----------
    N : int
        Nombre de fonctions de base (NBASIS)
    x : array
        Points d'évaluation
        
    Retourne:
    --------
    phi : array (N, len(x))
        Valeurs des fonctions de base
    phi_x : array (N, len(x))
        Dérivées premières
    phi_xx : array (N, len(x))
        Dérivées secondes
    """
    n_points = len(x)
    phi = np.zeros((N, n_points))
    phi_x = np.zeros((N, n_points))
    phi_xx = np.zeros((N, n_points))
    
    for i in range(n_points):
        xi = x[i]
        
        if np.abs(xi) < 1:
            # Points intérieurs - calcul via arccos
            T = np.arccos(xi)
            C = np.cos(T)
            S = np.sin(T)
            
            for j in range(N):
                n = j + 1  # degré du polynôme de Tchebychev
                TN = np.cos(n * T)
                TNT = -n * np.sin(n * T)
                TNTT = -n * n * TN
                
                # Conversion des dérivées par rapport à T en dérivées par rapport à x
                TNX = -TNT / S
                TNXX = TNTT / (S * S) - TNT * C / (S * S * S)
                
                # Construction des fonctions de base qui s'annulent aux bords
                if n % 2 == 0:
                    # N pair : φ = T_n(x) - 1
                    phi[j, i] = TN - 1
                    phi_x[j, i] = TNX
                else:
                    # N impair : φ = T_n(x) - x
                    phi[j, i] = TN - xi
                    phi_x[j, i] = TNX - 1
                
                phi_xx[j, i] = TNXX
        else:
            # Points aux bords (x = ±1)
            sgn = 1 if xi > 0 else -1
            
            for j in range(N):
                n = j + 1
                phi[j, i] = 0
                
                if n % 2 == 0:
                    phi_x[j, i] = sgn * n * n
                else:
                    phi_x[j, i] = n * n - 1
                
                phi_xx[j, i] = (sgn ** n) * n * n * (n * n - 1) / 3
    
    return phi, phi_x, phi_xx

In [2]:
### 2. Méthode de Collocation 

"""
collocation_method.py
Implémentation de la méthode de collocation pseudospectrale
"""

import numpy as np
from scipy.linalg import solve
from chebyshev_base import (
    chebyshev_points, create_boundary_function, create_basis_functions
)


def solve_bvp_collocation(N, d0, d1, d2, f, alpha, beta, x_plot=None):
    """
    Résout d2(x)*u_xx + d1(x)*u_x + d0(x)*u = f(x) sur [-1,1]
    avec u(-1)=alpha, u(1)=beta par collocation de Tchebychev
    (Approche similaire au code Fortran)
    
    Paramètres:
    -----------
    N : int
        Nombre de fonctions de base (NBASIS = N-2 dans la notation Fortran)
        Soit N total = NBASIS + 2
    d0, d1, d2 : callable
        Fonctions coefficients
    f : callable
        Terme source
    alpha, beta : float
        Conditions aux limites u(-1)=alpha, u(1)=beta
    x_plot : array, optional
        Points pour l'évaluation de la solution
        
    Retourne:
    --------
    x : array
        Points de collocation
    u : array
        Solution aux points de collocation
    coeffs : array
        Coefficients du développement
    """
    NBASIS = N  # Nombre de fonctions de base
    N_total = NBASIS + 2  # Nombre total de points (incluant les bords)
    
    # 1. Points de collocation (points intérieurs)
    # Utilisation des points de Gauss (racines de T_{NBASIS+1})
    x_colloc = chebyshev_points(NBASIS, 'gauss')
    
    # 2. Construction du système linéaire H * a = G
    H = np.zeros((NBASIS, NBASIS))
    G = np.zeros(NBASIS)
    
    # Évaluation des fonctions de base aux points de collocation
    # (calcul une fois pour tous)
    phi, phi_x, phi_xx = create_basis_functions(NBASIS, x_colloc)
    
    for i in range(NBASIS):
        xi = x_colloc[i]
        
        # Fonction de bord
        B, Bx = create_boundary_function(alpha, beta, np.array([xi]))
        B = B[0]
        Bx = Bx[0]
        
        # Second membre modifié: G = f - d0*B - d1*Bx
        G[i] = f(xi) - d0(xi) * B - d1(xi) * Bx
        
        # Coefficients de l'opérateur au point xi
        dd0 = d0(xi)
        dd1 = d1(xi)
        dd2 = d2(xi)
        
        # Remplissage de la matrice H
        for j in range(NBASIS):
            H[i, j] = dd2 * phi_xx[j, i] + dd1 * phi_x[j, i] + dd0 * phi[j, i]
    
    # 3. Résolution du système linéaire
    try:
        a = solve(H, G)  # Coefficients APHI
    except np.linalg.LinAlgError:
        print("Erreur : matrice singulière")
        a = np.zeros(NBASIS)
    
    # 4. Reconstruction de la solution aux points demandés
    if x_plot is None:
        x_plot = np.linspace(-1, 1, 101)
    
    u_plot = np.zeros_like(x_plot)
    B_plot, _ = create_boundary_function(alpha, beta, x_plot)
    u_plot = B_plot.copy()
    
    # Évaluation de la contribution des fonctions de base
    phi_plot, _, _ = create_basis_functions(NBASIS, x_plot)
    for j in range(NBASIS):
        u_plot += a[j] * phi_plot[j, :]
    
    # Solution aux points de collocation (pour comparaison)
    u_colloc = np.zeros(NBASIS)
    B_colloc, _ = create_boundary_function(alpha, beta, x_colloc)
    u_colloc = B_colloc.copy()
    for j in range(NBASIS):
        u_colloc += a[j] * phi[j, :]
    
    return x_colloc, u_colloc, a, x_plot, u_plot


def solve_bvp_collocation_matrix(N, d0, d1, d2, f, alpha, beta):
    """
    Version alternative utilisant la matrice de différenciation
    (Approche plus moderne et compacte)
    
    Paramètres:
    -----------
    N : int
        Nombre total de points (incluant les bords)
    d0, d1, d2 : callable
        Fonctions coefficients
    f : callable
        Terme source
    alpha, beta : float
        Conditions aux limites u(-1)=alpha, u(1)=beta
        
    Retourne:
    --------
    x : array
        Points de collocation (N+1 points)
    u : array
        Solution aux points de collocation
    """
    from chebyshev_base import chebyshev_matrix
    
    # Points de collocation incluant les bords
    D, x = chebyshev_matrix(N)
    D2 = D @ D
    
    # Évaluation des coefficients
    d0_vals = d0(x)
    d1_vals = d1(x)
    d2_vals = d2(x)
    f_vals = f(x)
    
    # Construction de l'opérateur L = d2*D2 + d1*D + d0*I
    L = np.diag(d2_vals) @ D2 + np.diag(d1_vals) @ D + np.diag(d0_vals)
    
    # Application des conditions aux limites
    L[0, :] = 0
    L[0, 0] = 1
    L[-1, :] = 0
    L[-1, -1] = 1
    
    F = f_vals.copy()
    F[0] = beta   # u(1) = beta
    F[-1] = alpha # u(-1) = alpha
    
    # Résolution
    u = solve(L, F)
    
    return x, u

In [3]:
### 3. Méthode de Galerkin

"""
galerkin_method.py
Implémentation de la méthode de Galerkin
"""

import numpy as np
from scipy.integrate import quad
from scipy.linalg import solve
from chebyshev_base import evaluate_chebyshev


def galerkin_weight(x):
    """
    Poids pour l'orthogonalité des polynômes de Tchebychev
    w(x) = 1/sqrt(1-x^2)
    """
    return 1.0 / np.sqrt(1.0 - x**2)


def integrate_galerkin(f, a=-1, b=1):
    """
    Intégrale de Galerkin avec poids de Tchebychev
    """
    def integrand(x):
        return f(x) * galerkin_weight(x)
    return quad(integrand, a, b, epsabs=1e-12, epsrel=1e-12)[0]


def solve_bvp_galerkin(N, d0, d1, d2, f, alpha, beta):
    """
    Résolution par méthode de Galerkin
    Utilise une base adaptée φ_k qui satisfait les conditions aux limites
    """
    NBASIS = N
    
    # Base adaptée: ψ_k(x) = T_{k+2}(x) - T_k(x) (s'annule en x=±1)
    # Pour simplifier, on utilise une approche avec fonctions de base classiques
    # et on impose les conditions par la méthode de Galerkin classique
    
    # Matrice de masse et matrice de rigidité
    M = np.zeros((NBASIS, NBASIS))
    K = np.zeros((NBASIS, NBASIS))
    F = np.zeros(NBASIS)
    
    # Pour simplifier, on suppose d2=1, d1=0, d0=0 (exemple)
    # Dans un cas général, les intégrales sont plus complexes
    
    # On utilise une quadrature de Gauss-Chebyshev
    from numpy.polynomial.chebyshev import chebgauss
    
    points, weights = chebgauss(NBASIS * 2)
    
    # Transformation sur [-1, 1]
    for i in range(NBASIS):
        # Fonction de test = T_i (sans adaptation)
        # Pour un vrai problème de Galerkin, il faudrait une base adaptée
        pass
    
    # Placeholder - à compléter selon le problème spécifique
    print("Méthode de Galerkin : implémentation simplifiée")
    print("Pour une implémentation complète, définir la base adaptée")
    
    return np.linspace(-1, 1, 100), np.zeros(100)


def create_galerkin_basis(N):
    """
    Crée une base de Galerkin adaptée aux conditions de Dirichlet homogènes
    φ_k(x) = T_{k+2}(x) - T_k(x)
    """
    def basis_function(k, x):
        Tk = np.polynomial.Chebyshev.basis(k)
        Tk2 = np.polynomial.Chebyshev.basis(k + 2)
        return Tk2(x) - Tk(x)
    
    return basis_function

In [4]:
### 4. Méthode Tau de Lanczos

"""
tau_method.py
Implémentation de la méthode Tau de Lanczos
"""

import numpy as np
from scipy.linalg import solve
from chebyshev_base import chebyshev_derivative


def solve_bvp_tau(N, d0, d1, d2, f, alpha, beta):
    """
    Résolution par méthode Tau de Lanczos
    
    On cherche u_N(x) = Σ_{k=0}^N a_k T_k(x)
    On projette l'équation sur les T_j pour j=0,...,N-2
    et on impose les conditions aux limites pour les 2 dernières équations
    """
    NBASIS = N + 1  # Nombre de coefficients (degré N)
    
    # Matrice du système dans l'espace des coefficients
    # Pour un opérateur L = d2*u_xx + d1*u_x + d0*u
    # La projection donne: Σ_k a_k <L T_k, T_j> = <f, T_j>
    
    # Construction de la matrice de transformation (opérateur dans l'espace spectral)
    # Pour simplifier, on traite le cas d2=1, d1=0, d0=0 (Laplacien)
    # u_xx = f avec u(±1)=alpha, beta
    
    # Dans l'espace spectral, T_k'' a une expression simple:
    # T_k'' = Σ_{m} c_{k,m} T_m avec c_{k,m} non nul pour m=k-2, k, k+2...
    
    # Matrice du système
    A = np.zeros((NBASIS + 2, NBASIS))  # N+1 équations + 2 CL
    B = np.zeros(NBASIS + 2)
    
    # Équations de projection (N-1 équations pour k=0 à N-2)
    for j in range(N - 1):
        # <u_xx, T_j> avec poids de Tchebychev
        # Pour le Laplacien: <T_k'', T_j> = ...
        # On utilise la propriété d'orthogonalité
        for k in range(NBASIS):
            # Calcul des produits scalaires
            if k == j:
                A[j, k] = 1.0  # Placeholder
        B[j] = 0.0  # <f, T_j>
    
    # Conditions aux limites
    # u(1) = Σ a_k T_k(1) = Σ a_k = beta
    # u(-1) = Σ a_k (-1)^k = alpha
    A[N-1, :] = [1.0] * NBASIS
    B[N-1] = beta
    A[N, :] = [(-1)**k for k in range(NBASIS)]
    B[N] = alpha
    
    # Résolution (système surdéterminé)
    # On utilise les moindres carrés ou on résout le système carré
    # en éliminant les dernières équations
    
    # Version simplifiée
    A_square = A[:NBASIS, :]
    B_square = B[:NBASIS]
    
    try:
        a = solve(A_square, B_square)
    except np.linalg.LinAlgError:
        a = np.linalg.lstsq(A_square, B_square, rcond=None)[0]
    
    # Reconstruction de la solution
    x_plot = np.linspace(-1, 1, 101)
    u_plot = np.zeros_like(x_plot)
    
    from chebyshev_base import evaluate_chebyshev
    for i, xi in enumerate(x_plot):
        u_plot[i] = evaluate_chebyshev(xi, a)
    
    return x_plot, u_plot

In [5]:
### 5. Méthode de Collocation pour problèmes non-linéaires


"""
nonlinear_collocation.py
Méthode de collocation pour problèmes non-linéaires avec Newton
"""

import numpy as np
from scipy.linalg import solve
from chebyshev_base import chebyshev_diff_matrix, create_boundary_function


def solve_nonlinear_newton(N, f_nonlinear, alpha, beta, 
                           y_init=None, max_iter=50, tol=1e-10):
    """
    Résout y'' = f(x, y, y') avec y(-1)=alpha, y(1)=beta
    par collocation de Tchebychev + méthode de Newton
    
    Paramètres:
    -----------
    N : int
        Nombre de points (incluant les bords)
    f_nonlinear : callable
        Fonction f(x, y, yp) retournant y''
    alpha, beta : float
        Conditions aux limites
    y_init : array, optional
        Initialisation (None = solution linéaire)
    max_iter : int
        Nombre maximum d'itérations Newton
    tol : float
        Tolérance de convergence
        
    Retourne:
    --------
    x : array
        Points de collocation
    y : array
        Solution
    """
    # Matrices de différenciation
    D, x = chebyshev_matrix(N)
    D2 = D @ D
    
    # Initialisation
    if y_init is None:
        # Solution linéaire entre alpha et beta
        y_init = alpha * (1 - x) / 2 + beta * (1 + x) / 2
    
    y = y_init.copy()
    
    for iteration in range(max_iter):
        # Calcul des dérivées
        yp = D @ y
        ypp = D2 @ y
        
        # Résidu: R = ypp - f(x, y, yp)
        R = ypp.copy()
        for i in range(N + 1):
            R[i] -= f_nonlinear(x[i], y[i], yp[i])
        
        # Jacobienne: J = D2 - df/dy - df/dyp * D
        # Approximation par différences finies
        J = D2.copy()
        eps = 1e-8
        
        for i in range(N + 1):
            # Perturbation pour df/dy
            y_pert = y.copy()
            y_pert[i] += eps
            yp_pert = D @ y_pert
            
            f_pert = f_nonlinear(x[i], y_pert[i], yp_pert[i])
            f_current = f_nonlinear(x[i], y[i], yp[i])
            
            df_dy = (f_pert - f_current) / eps
            J[i, i] -= df_dy
            
            # Perturbation pour df/dyp
            # On modifie légèrement y pour affecter yp
            # Cette partie est simplifiée - pour un calcul exact, 
            # il faudrait une perturbation sur yp directement
            # qui est plus complexe à implémenter
        
        # Application des conditions aux limites
        # y(1) = beta
        J[0, :] = 0
        J[0, 0] = 1
        R[0] = y[0] - beta
        
        # y(-1) = alpha
        J[-1, :] = 0
        J[-1, -1] = 1
        R[-1] = y[-1] - alpha
        
        # Mise à jour Newton
        try:
            delta = solve(J, -R)
        except np.linalg.LinAlgError:
            print(f"Matrice singulière à l'itération {iteration}")
            break
        
        y = y + delta
        
        # Vérification de convergence
        if np.linalg.norm(delta, np.inf) < tol:
            print(f"Convergence Newton en {iteration + 1} itérations")
            break
    
    return x, y


def solve_nonlinear_newton_exact_jacobian(N, f_nonlinear, df_dy, df_dyp,
                                           alpha, beta, y_init=None, max_iter=50, tol=1e-10):
    """
    Version avec jacobienne analytique exacte
    
    Paramètres supplémentaires:
   
    df_dy : callable
        Dérivée partielle ∂f/∂y
    df_dyp : callable
        Dérivée partielle ∂f/∂y'
    """
    D, x = chebyshev_matrix(N)
    D2 = D @ D
    
    if y_init is None:
        y_init = alpha * (1 - x) / 2 + beta * (1 + x) / 2
    
    y = y_init.copy()
    
    for iteration in range(max_iter):
        yp = D @ y
        ypp = D2 @ y
        
        # Résidu
        R = ypp.copy()
        for i in range(N + 1):
            R[i] -= f_nonlinear(x[i], y[i], yp[i])
        
        # Jacobienne analytique
        J = D2.copy()
        for i in range(N + 1):
            dfdy_val = df_dy(x[i], y[i], yp[i])
            dfdyp_val = df_dyp(x[i], y[i], yp[i])
            
            J[i, i] -= dfdy_val
            # Contribution de df/dyp: soustraire df/dyp * D
            # Cette ligne est modifiée pour inclure l'effet complet
            for j in range(N + 1):
                J[i, j] -= dfdyp_val * D[i, j]
        
        # Conditions aux limites
        J[0, :] = 0
        J[0, 0] = 1
        R[0] = y[0] - beta
        
        J[-1, :] = 0
        J[-1, -1] = 1
        R[-1] = y[-1] - alpha
        
        delta = solve(J, -R)
        y = y + delta
        
        if np.linalg.norm(delta, np.inf) < tol:
            print(f"Convergence en {iteration + 1} itérations")
            break
    
    return x, y

### 6. Exemples et tests complets

In [6]:
import numpy as np
import matplotlib.pyplot as plt
# A- (Puisse que je loge toutes les fonctions dans un meme fichier:"chebyshev_base")
# B- (On peut aussi loger  toutes les fonctions dans un differents fichiers:"chebyshev_base,collocation_methode,nonlinear_collocation...")


from chebyshev_base import chebyshev_points, chebyshev_diff_matrix
# A
from chebyshev_base import solve_bvp_collocation_basis, solve_bvp_collocation_matrix #A
from chebyshev_base import solve_nonlinear_newton, solve_nonlinear_newton_exact_jacobian #A
# B
#from collocation_method import solve_bvp_collocation_basis, solve_bvp_collocation_matrix #B
#from nonlinear_collocation import solve_nonlinear_newton, solve_nonlinear_newton_exact_jacobian #B



# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

In [7]:
# EXEMPLE 1: Équation linéaire avec solution exacte
# y'' + y = 0, y(-1)=0, y(1)=1
# Solution exacte: y(x) = sin(x+1)/sin(2)


def example_linear_oscillator():
    print("\n" + "="*60)
    print("EXEMPLE 1: Équation de l'oscillateur linéaire")
    print("y'' + y = 0, y(-1)=0, y(1)=1")
    print("="*60)
    
    # Définition des coefficients
    def d0(x): return 1.0
    def d1(x): return 0.0
    def d2(x): return 1.0
    def f(x): return 0.0
    
    alpha = 0.0  # y(-1) = 0
    beta = 1.0   # y(1) = 1
    
    # Solution exacte pour validation
    def exact_solution(x):
        return np.sin(x + 1) / np.sin(2)
    
    # Test avec différentes valeurs de N
    N_values = [5, 10, 15, 20, 25, 30]
    errors = []
    
    plt.figure(figsize=(14, 5))
    
    # Sous-plot: solutions pour différents N
    plt.subplot(1, 2, 1)
    x_ref = np.linspace(-1, 1, 200)
    y_ref = exact_solution(x_ref)
    plt.plot(x_ref, y_ref, 'k-', linewidth=2, label='Solution exacte', alpha=0.7)
    
    for N in N_values[::2]:  # Tous les 2 pour la clarté
        x_colloc, u_colloc, coeffs, x_plot, u_plot = solve_bvp_collocation(
            N, d0, d1, d2, f, alpha, beta, x_ref
        )
        plt.plot(x_plot, u_plot, '--', linewidth=1.5, label=f'N = {N}')
    
    plt.xlabel('x')
    plt.ylabel('y(x)')
    plt.title('Solutions pour différents N')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Sous-plot: convergence spectrale
    plt.subplot(1, 2, 2)
    
    for N in N_values:
        _, u_colloc, _, x_plot, u_plot = solve_bvp_collocation(
            N, d0, d1, d2, f, alpha, beta, x_plot=x_ref
        )
        error = np.max(np.abs(u_plot - exact_solution(x_ref)))
        errors.append(error)
        print(f"N = {N:2d} : Erreur max = {error:.2e}")
    
    plt.semilogy(N_values, errors, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Nombre de fonctions de base (N)')
    plt.ylabel('Erreur maximale')
    plt.title('Convergence spectrale (échelle log)')
    plt.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.show()
    
    # Version avec matrice de différenciation
    print("\n" + "-"*40)
    print("Version avec matrice de différenciation:")
    N_total = 30
    x, u = solve_bvp_collocation_matrix(N_total, d0, d1, d2, f, alpha, beta)
    
    plt.figure(figsize=(10, 5))
    plt.plot(x_ref, exact_solution(x_ref), 'k-', linewidth=2, label='Exacte')
    plt.plot(x, u, 'ro', markersize=5, label='Collocation (matrice D)')
    plt.xlabel('x')
    plt.ylabel('y(x)')
    plt.title('Méthode de collocation avec matrice de différenciation')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [8]:
# EXEMPLE 2: Équation de Bratu (non-linéaire)
# y'' + λ e^y = 0, y(-1)=0, y(1)=0


def example_bratu():
    print("\n" + "="*60)
    print("EXEMPLE 2: Équation de Bratu (non-linéaire)")
    print("y'' + λ exp(y) = 0, y(-1)=0, y(1)=0")
    print("="*60)
    
    lambda_param = 3.0  # Paramètre de non-linéarité
    
    # Définition de la non-linéarité y'' = f(x, y, y')
    def f_nonlinear(x, y, yp):
        return -lambda_param * np.exp(y)
    
    # Dérivées pour la jacobienne analytique
    def df_dy(x, y, yp):
        return -lambda_param * np.exp(y)
    
    def df_dyp(x, y, yp):
        return 0.0
    
    alpha = 0.0
    beta = 0.0
    
    # Résolution pour différentes valeurs de λ
    lambda_values = [1.0, 2.0, 3.0, 3.5]
    
    plt.figure(figsize=(12, 6))
    
    for lam in lambda_values:
        def f_bratu(x, y, yp):
            return -lam * np.exp(y)
        
        x, y = solve_nonlinear_newton(30, f_bratu, alpha, beta, max_iter=30)
        plt.plot(x, y, 'o-', linewidth=2, markersize=4, label=f'λ = {lam}')
        print(f"λ = {lam}: max y = {np.max(y):.4f}")
    
    plt.xlabel('x')
    plt.ylabel('y(x)')
    plt.title('Équation de Bratu: y\'\' + λ exp(y) = 0')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Étude de convergence
    print("\n" + "-"*40)
    print("Étude de convergence:")
    
    N_values = [10, 15, 20, 25, 30, 35, 40]
    solutions = {}
    
    for N in N_values:
        x, y = solve_nonlinear_newton(N, f_bratu, alpha, beta, max_iter=50)
        solutions[N] = (x, y)
    
    # Solution de référence (N=50)
    x_ref, y_ref = solve_nonlinear_newton(50, f_bratu, alpha, beta, max_iter=50)
    
    errors = []
    for N in N_values:
        x_N, y_N = solutions[N]
        # Interpolation sur la grille de référence
        y_interp = np.interp(x_ref, x_N, y_N)
        error = np.max(np.abs(y_interp - y_ref))
        errors.append(error)
        print(f"N = {N:2d} : Erreur max = {error:.2e}")
    
    plt.figure(figsize=(10, 5))
    plt.semilogy(N_values, errors, 'ro-', linewidth=2, markersize=8)
    plt.xlabel('Nombre de points (N)')
    plt.ylabel('Erreur maximale')
    plt.title('Convergence spectrale - Équation de Bratu')
    plt.grid(True, alpha=0.3, which='both')
    plt.show()

In [9]:
# EXEMPLE 3: Équation de Duffing (oscillateur non-linéaire)
# y'' + ω² y + ε y³ = 0, y(-1)=0, y(1)=1


def example_duffing():
    print("\n" + "="*60)
    print("EXEMPLE 3: Équation de Duffing (oscillateur non-linéaire)")
    print("y'' + ω² y + ε y³ = 0, y(-1)=0, y(1)=1")
    print("="*60)
    
    omega = 5.0
    epsilon = 10.0
    
    def f_duffing(x, y, yp):
        return -omega**2 * y - epsilon * y**3
    
    alpha = 0.0
    beta = 1.0
    
    # Comparaison linéaire vs non-linéaire
    x_lin, y_lin = solve_nonlinear_newton(30, lambda x, y, yp: -omega**2 * y, 
                                          alpha, beta, max_iter=30)
    x_nonlin, y_nonlin = solve_nonlinear_newton(30, f_duffing, alpha, beta, max_iter=30)
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(x_lin, y_lin, 'b-', linewidth=2, label='Linéaire (ε=0)')
    plt.plot(x_nonlin, y_nonlin, 'r-', linewidth=2, label=f'Non-linéaire (ε={epsilon})')
    plt.xlabel('x')
    plt.ylabel('y(x)')
    plt.title(f'Équation de Duffing: ω = {omega}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Effet du paramètre ε
    plt.subplot(1, 2, 2)
    eps_values = [0, 5, 10, 20]
    
    for eps in eps_values:
        def f_eps(x, y, yp):
            return -omega**2 * y - eps * y**3
        x, y = solve_nonlinear_newton(30, f_eps, alpha, beta, max_iter=30)
        plt.plot(x, y, '-', linewidth=2, label=f'ε = {eps}')
    
    plt.xlabel('x')
    plt.ylabel('y(x)')
    plt.title(f'Effet du paramètre ε (ω = {omega})')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [10]:
# EXEMPLE 4: Problème avec conditions mixtes (Neumann + Dirichlet)
# y'' + y' + y² = cos(πx), y'(-1)=0, y(1)=0

def example_mixed_bc():
    print("\n" + "="*60)
    print("EXEMPLE 4: Conditions mixtes (Neumann + Dirichlet)")
    print("y'' + y' + y² = cos(πx), y'(-1)=0, y(1)=0")
    print("="*60)
    
    from chebyshev_base import chebyshev_matrix
    
    def f_nonlinear(x, y, yp):
        return -yp - y**2 + np.cos(np.pi * x)
    
    # Pour les conditions mixtes, on utilise une approche avec matrice de différenciation
    N = 30
    D, x = chebyshev_matrix(N)
    D2 = D @ D
    
    # Initialisation
    y = np.zeros(N + 1)
    
    max_iter = 50
    tol = 1e-10
    
    for iteration in range(max_iter):
        yp = D @ y
        ypp = D2 @ y
        
        # Résidu
        R = ypp.copy()
        for i in range(N + 1):
            R[i] -= f_nonlinear(x[i], y[i], yp[i])
        
        # Jacobienne approchée
        J = D2 + D  # Partie linéaire
        for i in range(N + 1):
            J[i, i] += 2 * y[i]  # Dérivée de y²
        
        # Condition de Neumann à x = -1 (dernière ligne)
        J[-1, :] = D[-1, :]
        R[-1] = yp[-1] - 0
        
        # Condition de Dirichlet à x = 1 (première ligne)
        J[0, :] = 0
        J[0, 0] = 1
        R[0] = y[0] - 0
        
        # Mise à jour
        from scipy.linalg import solve
        delta = solve(J, -R)
        y = y + delta
        
        if np.linalg.norm(delta, np.inf) < tol:
            print(f"Convergence en {iteration + 1} itérations")
            break
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(x, y, 'ro-', linewidth=2, markersize=4)
    plt.xlabel('x')
    plt.ylabel('y(x)')
    plt.title('Solution y(x)')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    yp = D @ y
    plt.plot(x, yp, 'bo-', linewidth=2, markersize=4)
    plt.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    plt.xlabel('x')
    plt.ylabel('y\'(x)')
    plt.title('Dérivée y\'(x) - condition de Neumann à x=-1')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Vérification: y'(-1) = {yp[-1]:.2e} (attendu: 0)")
    print(f"Vérification: y(1) = {y[0]:.2e} (attendu: 0)")

# Comparaison des méthodes (Galerkin, Tau, Collocation)

In [11]:
def example_comparison():
    print("\n" + "="*60)
    print("EXEMPLE 5: Comparaison des méthodes spectrales")
    print("Problème: y'' + y = cos(πx), y(-1)=0, y(1)=0")
    print("="*60)
    
    # Définition du problème
    def d0(x): return 1.0
    def d1(x): return 0.0
    def d2(x): return 1.0
    def f(x): return np.cos(np.pi * x)
    
    alpha = 0.0
    beta = 0.0
    N = 20
    
    # 1. Méthode de collocation avec matrice de différenciation
    print("\n1. Résolution par collocation (matrice de différenciation)...")
    x_colloc_mat, u_colloc_mat = solve_bvp_collocation_matrix(N, d0, d1, d2, f, alpha, beta)
    print(f"   Points: {len(x_colloc_mat)}")
    
    # 2. Méthode de collocation avec fonctions de base (style Fortran)
    print("\n2. Résolution par collocation (fonctions de base)...")
    x_ref = np.linspace(-1, 1, 200)
    x_colloc_base, u_colloc_base, coeffs, x_plot, u_basis = solve_bvp_collocation_basis(
        N, d0, d1, d2, f, alpha, beta, x_ref
    )
    print(f"   Points de collocation: {len(x_colloc_base)}")
    print(f"   Points d'évaluation: {len(x_plot)}")
    
    # 3. Méthode de Tau (version simplifiée)
    print("\n3. Résolution par méthode Tau...")
    x_tau, u_tau = solve_bvp_tau(N, d0, d1, d2, f, alpha, beta)
    print(f"   Points: {len(x_tau)}")
    
    # Visualisation des résultats
    plt.figure(figsize=(14, 5))
    
    # Sous-plot 1: Comparaison des solutions
    plt.subplot(1, 2, 1)
    plt.plot(x_colloc_mat, u_colloc_mat, 'ro', markersize=5, label='Collocation (matrice)', alpha=0.7)
    plt.plot(x_plot, u_basis, 'b-', linewidth=2, label='Collocation (fonctions base)')
    plt.plot(x_tau, u_tau, 'g--', linewidth=2, label='Tau')
    plt.xlabel('x', fontsize=12)
    plt.ylabel('y(x)', fontsize=12)
    plt.title('Comparaison des méthodes spectrales', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Sous-plot 2: Différences entre méthodes
    plt.subplot(1, 2, 2)
    
    # Interpolation pour comparaison sur la même grille
    u_colloc_mat_interp = np.interp(x_plot, x_colloc_mat, u_colloc_mat)
    u_tau_interp = np.interp(x_plot, x_tau, u_tau)
    
    diff_matrix_basis = np.abs(u_colloc_mat_interp - u_basis)
    diff_tau_basis = np.abs(u_tau_interp - u_basis)
    
    plt.semilogy(x_plot, diff_matrix_basis, 'r-', linewidth=2, label='Collocation (matrice vs base)')
    plt.semilogy(x_plot, diff_tau_basis, 'b-', linewidth=2, label='Tau vs base')
    plt.xlabel('x', fontsize=12)
    plt.ylabel('Différence absolue', fontsize=12)
    plt.title('Comparaison des différences (échelle log)', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.show()
    
    # Affichage des erreurs
    print("\n" + "-"*40)
    print("Résumé des erreurs:")
    print(f"Erreur max Collocation (matrice vs base): {np.max(diff_matrix_basis):.2e}")
    print(f"Erreur max Tau vs base: {np.max(diff_tau_basis):.2e}")
    
    # Vérification des conditions aux limites
    print("\nVérification des conditions aux limites:")
    print(f"  u(-1) attendu = {alpha}, obtenu: Collocation matrice = {u_colloc_mat[-1]:.2e}, "
          f"Collocation base = {u_basis[0]:.2e}, Tau = {u_tau[-1]:.2e}")
    print(f"  u(1) attendu = {beta}, obtenu: Collocation matrice = {u_colloc_mat[0]:.2e}, "
          f"Collocation base = {u_basis[-1]:.2e}, Tau = {u_tau[0]:.2e}")

## Résumé des codes fournis

| Fichier | Description |
|---------|-------------|
| `chebyshev_base.py` | Fonctions de base : points de collocation, matrices de différenciation, fonctions de bord, fonctions de base φ_j |
| `collocation_method.py` | Implémentation de la méthode de collocation (2 versions) |
| `galerkin_method.py` | Implémentation de la méthode de Galerkin |
| `tau_method.py` | Implémentation de la méthode Tau de Lanczos |
| `nonlinear_collocation.py` | Méthode de collocation pour problèmes non-linéaires avec Newton |
| `examples.py` | Exemples complets pour toutes les méthodes |

In [ ]:
# Comparaison des méthodes (Galerkin, Tau, Collocation)

def example_comparison():
    print("\n" + "="*60)
    print("EXEMPLE 5: Comparaison des méthodes spectrales")
    print("Problème: y'' + y = cos(πx), y(-1)=0, y(1)=0")
    print("="*60)
    
    from scipy.linalg import solve
    
    def d0(x): return 1.0
    def d1(x): return 0.0
    def d2(x): return 1.0
    def f(x): return np.cos(np.pi * x)
    
    alpha = 0.0
    beta = 0.0
    N = 20
    
    # Méthode de collocation
    from collocation_method import solve_bvp_collocation_matrix
    x_colloc, u_colloc = solve_bvp_collocation_matrix(N, d0, d1, d2, f, alpha, beta)
    
    # Méthode de collocation 
    x_ref = np.linspace(-1, 1, 200)
    _, _, _, x_plot, u_basis = solve_bvp_collocation(N, d0, d1, d2, f, alpha, beta, x_ref)
    
    # Méthode de Tau
    from tau_method import solve_bvp_tau
    x_tau, u_tau = solve_bvp_tau(N, d0, d1, d2, f, alpha, beta)
    
    # Visualisation
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(x_colloc, u_colloc, 'ro', markersize=6, label='Collocation (matrice)')
    plt.plot(x_plot, u_basis, 'b-', linewidth=2, label='Collocation (fonctions base)')
    plt.plot(x_tau, u_tau, 'g--', linewidth=2, label='Tau')
    plt.xlabel('x')
    plt.ylabel('y(x)')
    plt.title('Comparaison des méthodes')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Différences entre méthodes
    plt.subplot(1, 2, 2)
    # Interpolation pour comparaison
    u_colloc_interp = np.interp(x_plot, x_colloc, u_colloc)
    diff_matrix_basis = np.abs(u_colloc_interp - u_basis)
    diff_tau_basis = np.abs(np.interp(x_plot, x_tau, u_tau) - u_basis)
    
    plt.semilogy(x_plot, diff_matrix_basis, 'r-', linewidth=2, label='Collocation (matrice vs base)')
    plt.semilogy(x_plot, diff_tau_basis, 'b-', linewidth=2, label='Tau vs base')
    plt.xlabel('x')
    plt.ylabel('Différence')
    plt.title('Comparaison des différences')
    plt.legend()
    plt.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Erreur max Collocation (matrice vs base): {np.max(diff_matrix_basis):.2e}")
    print(f"Erreur max Tau vs base: {np.max(diff_tau_basis):.2e}")



# Programme principal

if __name__ == "__main__":
    print("\n" + "="*60)
    print("RÉSOLUTION D'EDO PAR MÉTHODES SPECTRALES DE TCHEBYCHEV")
    print("="*60)
    
    # Exécution des exemples
    example_linear_oscillator()
    example_bratu()
    example_duffing()
    example_mixed_bc()
    example_comparison()
    
    print("\n" + "="*60)
    print("FIN DES EXEMPLES")
    print("="*60)
